# Diagnóstico de Dados e Análises — TCC REN 1.000/2021

Este notebook realiza:
1. **Diagnóstico de qualidade** dos CSVs brutos da ANEEL
2. **Análises estatísticas** para o TCC (pré vs pós REN 1000)
3. **Recomendações** de melhorias no dashboard

---

In [1]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # backend não-interativo para nbconvert
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from statsmodels.tsa.seasonal import seasonal_decompose

# Forçar strings como object (evita problemas com pyarrow no pandas 3.x)
try:
    pd.options.mode.string_storage = 'python'
except Exception:
    pass  # versões mais antigas do pandas

# Configuração visual
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'figure.dpi': 100,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'font.size': 10,
    'axes.grid': True,
    'grid.alpha': 0.3,
})
sns.set_style('whitegrid')

# Caminhos
ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent

RAW = ROOT / 'data' / 'raw'
PROC = ROOT / 'data' / 'processed'
ANALYSIS = PROC / 'analysis'

print(f'ROOT: {ROOT}')
print(f'RAW existe: {RAW.exists()}')
print(f'ANALYSIS existe: {ANALYSIS.exists()}')

ROOT: /home/gianmarinolc/Documents/Estudos/TCC_leo_main
RAW existe: True
ANALYSIS existe: True


---
## PARTE 1 — Diagnóstico de Qualidade dos Dados

### 1.1 Leitura robusta dos CSVs brutos

In [2]:
def ler_csv_robusto(caminho, nrows=None):
    """Lê CSV com fallback de encoding e separador."""
    encodings = ['utf-8', 'latin-1', 'cp1252', 'utf-16']
    separadores = [';', ',']
    
    for enc in encodings:
        for sep in separadores:
            try:
                df = pd.read_csv(caminho, encoding=enc, sep=sep,
                                 nrows=nrows, low_memory=False,
                                 dtype_backend='numpy_nullable')
                if len(df.columns) > 1:
                    df.columns = [c.strip().strip('"').strip() for c in df.columns]
                    print(f'  OK: encoding={enc}, sep="{sep}"')
                    return df
            except (UnicodeDecodeError, UnicodeError, pd.errors.ParserError):
                continue
    
    raise ValueError(f'Nao foi possivel ler {caminho}')


def is_text_col(series):
    """Verifica se coluna e texto."""
    return pd.api.types.is_string_dtype(series) or series.dtype == 'object'


def parse_numero_br(series):
    """Converte numero brasileiro (1.234,56) para float."""
    if is_text_col(series):
        txt = series.astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
        return pd.to_numeric(txt, errors='coerce')
    return pd.to_numeric(series, errors='coerce')


# Inventario de arquivos raw
arquivos_raw = {
    'qualidade_comercial': RAW / 'qualidade-atendimento-comercial.csv',
    'dominio_indicadores': RAW / 'dominio-indicadores.csv',
    'indger_comerciais': RAW / 'indger-dados-comerciais.csv',
}

servicos_csvs = sorted(RAW.glob('indger-dados-servicos-comerciais-*.csv'))
if servicos_csvs:
    arquivos_raw['indger_servicos_amostra'] = servicos_csvs[0]
    print(f'Total de CSVs mensais de servicos: {len(servicos_csvs)}')

for nome, caminho in arquivos_raw.items():
    existe = 'OK' if caminho.exists() else 'FALTA'
    tamanho = f'{caminho.stat().st_size / 1e6:.1f} MB' if caminho.exists() else 'NAO ENCONTRADO'
    print(f'{existe} {nome}: {tamanho}')

Total de CSVs mensais de servicos: 36
OK qualidade_comercial: 85.3 MB
OK dominio_indicadores: 0.1 MB
OK indger_comerciais: 106.8 MB
OK indger_servicos_amostra: 234.7 MB


In [3]:
# Ler cada arquivo (amostra de 50k linhas para rapidez)
datasets_raw = {}

for nome, caminho in arquivos_raw.items():
    if not caminho.exists():
        print(f'AVISO: {nome}: arquivo nao encontrado, pulando')
        continue
    print(f'\n--- Lendo {nome} ({caminho.name}) ---')
    try:
        n = None if 'dominio' in nome else 50_000
        datasets_raw[nome] = ler_csv_robusto(caminho, nrows=n)
        df = datasets_raw[nome]
        print(f'  Shape: {df.shape}')
        print(f'  Colunas: {list(df.columns)}')
    except Exception as e:
        print(f'ERRO ao ler {nome}: {e}')


--- Lendo qualidade_comercial (qualidade-atendimento-comercial.csv) ---
  OK: encoding=latin-1, sep=";"
  Shape: (50000, 7)
  Colunas: ['DatGeracaoConjuntoDados', 'SigAgente', 'NumCNPJ', 'SigIndicador', 'AnoIndice', 'NumPeriodoIndice', 'VlrIndiceEnviado']

--- Lendo dominio_indicadores (dominio-indicadores.csv) ---
ERRO ao ler dominio_indicadores: Nao foi possivel ler /home/gianmarinolc/Documents/Estudos/TCC_leo_main/data/raw/dominio-indicadores.csv

--- Lendo indger_comerciais (indger-dados-comerciais.csv) ---


  OK: encoding=latin-1, sep=";"
  Shape: (50000, 63)
  Colunas: ['DatGeracaoConjuntoDados', 'NumCNPJ', 'SigAgente', 'NomAgente', 'NomTipoOutorga', 'DatReferenciaInformada', 'CodMunicipioIBGE', 'QtdUCAtiva', 'QtdUCAtivaFat', 'QtdFatura', 'QtdFaturaSemLeitura', 'QtdFaturaSemLeituraImpAcesso', 'QtdFaturaSemLeituraEmergencia', 'QtdFaturaSemLeituraPlurimensa', 'QtdFaturaSemLeituraFatEstimad', 'QtdFaturaSemLeituraFimContrat', 'QtdFaturaSemLeituraAusenciaTm', 'QtdRefaturamento', 'QtdFaturaAcerto', 'QtdFaturaAcertoFatIncorreto', 'QtdFaturaAcertoFatIncorDevDob', 'QtdFaturaAcertoFatIncorDevSEC', 'QtdFaturaAcertoFatIncorDevSET', 'QtdFaturaAcertoFatImpAcesso', 'QtdFaturaAcertoFatEmergencia', 'QtdFaturaSemLeituraFatMedia', 'QtdFaturaSemLeituraFatCustoDi', 'QtdFaturaComLeitura', 'QtdFaturaComAutoleitura', 'QtdConsComLeituraPlurimensal', 'QtdConsComAutoleitura', 'QtdRestAnteDist', 'QtdRestAtrasadoDist', 'QtdRestPendenteDist', 'QtdRestPendenteAtrasado', 'VlrRestAnte', 'VlrRestAtrasado', 'VlrRestPenden

### 1.2 Analise de qualidade por dataset

In [4]:
def diagnostico_qualidade(df, nome):
    """Gera diagnostico completo de qualidade de um DataFrame."""
    print(f'\n{"=" * 70}')
    print(f'DIAGNOSTICO: {nome}')
    print(f'{"=" * 70}')
    
    print(f'\nDimensoes: {df.shape[0]:,} linhas x {df.shape[1]} colunas')
    print(f'Memoria: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
    
    nulos = df.isnull().sum()
    pct_nulos = (nulos / len(df) * 100).round(1)
    
    resultados = []
    print(f'\nNulos por coluna:')
    for col in df.columns:
        pct = float(pct_nulos[col])
        if pct == 0:
            status = 'OK'
        elif pct < 5:
            status = 'ATENCAO'
        elif pct < 50:
            status = 'ALERTA'
        else:
            status = 'CRITICO'
        resultados.append({
            'coluna': col, 'dtype': str(df[col].dtype),
            'nulos': int(nulos[col]), 'pct_nulo': pct,
            'status': status, 'n_unicos': int(df[col].nunique()),
        })
        print(f'  [{status}] {col}: {pct}% nulo | dtype={df[col].dtype} | {df[col].nunique()} unicos')
    
    text_cols = [c for c in df.columns if is_text_col(df[c])]
    if text_cols:
        print(f'\nColunas de texto ({len(text_cols)}):')
        for col in text_cols:
            n_unique = df[col].nunique()
            if n_unique <= 20:
                vals = df[col].value_counts().head(10)
                top_vals = ', '.join([f'{v}({c})' for v, c in vals.items()])
                print(f'  {col} [{n_unique} valores]: {top_vals}')
            else:
                print(f'  {col} [{n_unique} valores unicos]')
    
    num_cols = df.select_dtypes(include=[np.number]).columns
    if len(num_cols) > 0:
        print(f'\nColunas numericas ({len(num_cols)}):')
        for col in num_cols:
            s = df[col].dropna()
            if len(s) == 0:
                print(f'  {col}: VAZIA')
                continue
            neg = int((s < 0).sum())
            zeros = int((s == 0).sum())
            flag = ' NEGATIVOS!' if neg > 0 else ''
            print(f'  {col}: min={float(s.min()):.2f}, max={float(s.max()):.2f}, '
                  f'media={float(s.mean()):.2f}, zeros={zeros:,}{flag}')
    
    n_dup = int(df.duplicated().sum())
    if n_dup > 0:
        print(f'\nLinhas duplicadas: {n_dup:,} ({n_dup/len(df)*100:.1f}%)')
    else:
        print(f'\nSem linhas duplicadas')
    
    return pd.DataFrame(resultados)

In [5]:
relatorios = {}
for nome, df in datasets_raw.items():
    relatorios[nome] = diagnostico_qualidade(df, nome)


DIAGNOSTICO: qualidade_comercial

Dimensoes: 50,000 linhas x 7 colunas
Memoria: 12.7 MB

Nulos por coluna:
  [OK] DatGeracaoConjuntoDados: 0.0% nulo | dtype=string | 1 unicos
  [ATENCAO] SigAgente: 1.9% nulo | dtype=string | 99 unicos
  [OK] NumCNPJ: 0.0% nulo | dtype=Int64 | 101 unicos
  [OK] SigIndicador: 0.0% nulo | dtype=string | 96 unicos
  [OK] AnoIndice: 0.0% nulo | dtype=Int64 | 2 unicos
  [OK] NumPeriodoIndice: 0.0% nulo | dtype=Int64 | 12 unicos
  [OK] VlrIndiceEnviado: 0.0% nulo | dtype=string | 5868 unicos

Colunas de texto (4):
  DatGeracaoConjuntoDados [1 valores]: 05-02-2026(50000)
  SigAgente [99 valores unicos]
  SigIndicador [96 valores unicos]
  VlrIndiceEnviado [5868 valores unicos]

Colunas numericas (3):
  NumCNPJ: min=1229747000189.00, max=97839922000129.00, media=43118408893727.38, zeros=0
  AnoIndice: min=2011.00, max=2012.00, media=2011.57, zeros=0
  NumPeriodoIndice: min=1.00, max=12.00, media=5.96, zeros=0



Sem linhas duplicadas

DIAGNOSTICO: indger_comerciais

Dimensoes: 50,000 linhas x 63 colunas
Memoria: 69.2 MB

Nulos por coluna:
  [OK] DatGeracaoConjuntoDados: 0.0% nulo | dtype=string | 1 unicos
  [OK] NumCNPJ: 0.0% nulo | dtype=Int64 | 12 unicos
  [OK] SigAgente: 0.0% nulo | dtype=string | 12 unicos
  [OK] NomAgente: 0.0% nulo | dtype=string | 12 unicos
  [OK] NomTipoOutorga: 0.0% nulo | dtype=string | 2 unicos
  [OK] DatReferenciaInformada: 0.0% nulo | dtype=string | 36 unicos
  [OK] CodMunicipioIBGE: 0.0% nulo | dtype=Int64 | 1563 unicos
  [OK] QtdUCAtiva: 0.0% nulo | dtype=Int64 | 18986 unicos
  [OK] QtdUCAtivaFat: 0.0% nulo | dtype=Int64 | 18801 unicos
  [OK] QtdFatura: 0.0% nulo | dtype=Int64 | 19132 unicos
  [OK] QtdFaturaSemLeitura: 0.0% nulo | dtype=Int64 | 4221 unicos
  [OK] QtdFaturaSemLeituraImpAcesso: 0.0% nulo | dtype=Int64 | 1905 unicos
  [OK] QtdFaturaSemLeituraEmergencia: 0.0% nulo | dtype=Int64 | 543 unicos
  [OK] QtdFaturaSemLeituraPlurimensa: 0.0% nulo | dtype=In

  [OK] DthCarga: 0.0% nulo | dtype=string | 23 unicos

Colunas de texto (15):
  DatGeracaoConjuntoDados [1 valores]: 2026-02-05(50000)
  SigAgente [12 valores]: RGE(14040), Enel GO(8784), Neoenergia Elektro(8424), Copel-Dis(7649), Energisa MT(5088), Amazonas Energia(2232), EDP SP(1008), CPFL Piratininga(972), Energisa AC(792), Roraima Energia(540)
  NomAgente [12 valores]: RGE SUL DISTRIBUIDORA DE ENERGIA S.A.(14040), EQUATORIAL GOIÁS DISTRIBUIDORA DE ENERGIA S.A.(8784), ELEKTRO REDES S.A.(8424), COPEL DISTRIBUIÇÃO S.A.(7649), ENERGISA MATO GROSSO DISTRIBUIDORA DE ENERGIA S.A.(5088), AMAZONAS ENERGIA S.A.(2232), EDP SÃO PAULO DISTRIBUIÇÃO DE ENERGIA S.A.(1008), COMPANHIA PIRATININGA DE FORÇA E LUZ(972), ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A.(792), RORAIMA ENERGIA S.A.(540)
  NomTipoOutorga [2 valores]: Concessão(49892), Permissão(108)
  DatReferenciaInformada [36 valores unicos]
  VlrRestAnte [7075 valores unicos]
  VlrRestAtrasado [1838 valores unicos]
  VlrRestPendente [3450 v

  QtdFaturaSemLeituraFimContrat: min=0.00, max=5511.00, media=36.33, zeros=17,791
  QtdFaturaSemLeituraAusenciaTm: min=0.00, max=8268.00, media=34.90, zeros=14,975
  QtdRefaturamento: min=0.00, max=31118.00, media=58.52, zeros=5,834
  QtdFaturaAcerto: min=0.00, max=24754.00, media=56.73, zeros=15,474
  QtdFaturaAcertoFatIncorreto: min=0.00, max=24749.00, media=40.36, zeros=19,898
  QtdFaturaAcertoFatIncorDevDob: min=0.00, max=1444.00, media=3.99, zeros=32,537
  QtdFaturaAcertoFatIncorDevSEC: min=0.00, max=12371.00, media=23.69, zeros=25,235
  QtdFaturaAcertoFatIncorDevSET: min=0.00, max=12371.00, media=6.82, zeros=46,278
  QtdFaturaAcertoFatImpAcesso: min=0.00, max=1273.00, media=16.06, zeros=29,115
  QtdFaturaAcertoFatEmergencia: min=0.00, max=1075.00, media=0.41, zeros=48,542
  QtdFaturaSemLeituraFatMedia: min=0.00, max=49021.00, media=691.18, zeros=925
  QtdFaturaSemLeituraFatCustoDi: min=0.00, max=7372.00, media=71.47, zeros=5,086
  QtdFaturaComLeitura: min=0.00, max=878654.00, med


Nulos por coluna:
  [OK] DatGeracaoConjuntoDados: 0.0% nulo | dtype=string | 1 unicos
  [OK] NumCNPJ: 0.0% nulo | dtype=Int64 | 19 unicos
  [ATENCAO] SigAgente: 0.1% nulo | dtype=string | 18 unicos
  [ATENCAO] NomAgente: 0.1% nulo | dtype=string | 18 unicos
  [ATENCAO] NomTipoOutorga: 0.1% nulo | dtype=string | 2 unicos
  [OK] DatReferenciaInformada: 0.0% nulo | dtype=string | 1 unicos
  [OK] CodMunicipioIBGE: 0.0% nulo | dtype=Int64 | 518 unicos
  [OK] CodTipoServico: 0.0% nulo | dtype=Int64 | 99 unicos
  [OK] DscDispositivo: 0.0% nulo | dtype=string | 75 unicos
  [OK] DscPrazo: 0.0% nulo | dtype=string | 23 unicos


  [OK] DscTipoServico: 0.0% nulo | dtype=string | 99 unicos
  [OK] QtdServRealizado: 0.0% nulo | dtype=Int64 | 680 unicos
  [OK] MdaTempoMedServRealizado: 0.0% nulo | dtype=string | 1674 unicos
  [OK] QtdServRealizDescPrazo: 0.0% nulo | dtype=Int64 | 18 unicos
  [OK] MdaTempoMedAServRealzDescPrazo: 0.0% nulo | dtype=string | 170 unicos
  [OK] QtdServSolicitado: 0.0% nulo | dtype=Int64 | 680 unicos
  [CRITICO] QtdServAindaNaoRealiz: 91.4% nulo | dtype=Int64 | 95 unicos
  [CRITICO] QtdServSuspenso: 99.2% nulo | dtype=Int64 | 38 unicos
  [CRITICO] QtdServPendAtdDescPrazo: 99.2% nulo | dtype=Int64 | 23 unicos
  [OK] MdaAtrazoServPendAtdDescPrazo: 0.0% nulo | dtype=string | 246 unicos
  [OK] VlrPagoCompensacao: 0.0% nulo | dtype=string | 445 unicos
  [OK] DthCarga: 0.0% nulo | dtype=string | 70 unicos

Colunas de texto (13):
  DatGeracaoConjuntoDados [1 valores]: 2026-02-05(50000)
  SigAgente [18 valores]: Copel-Dis(38997), Celetro(2770), Cooperluz(1649), Certaja Energia(1635), Certhil(1584

### 1.3 Validacao de Schema

In [6]:
CONTRATOS = {
    'qualidade_comercial': {
        'colunas_obrigatorias': ['sigagente', 'sigindicador', 'anoindice',
                                 'numperiodoindice', 'vlrindiceenviado'],
        'range_anos': (2011, 2025), 'col_ano': 'anoindice',
    },
    'indger_comerciais': {
        'colunas_obrigatorias': ['datreferenciainformada', 'sigagente',
                                 'nomagente', 'qtducativa'],
        'col_data': 'datreferenciainformada',
    },
    'indger_servicos_amostra': {
        'colunas_obrigatorias': ['datreferenciainformada', 'sigagente', 'nomagente',
                                 'codmunicipioibge', 'codtiposervico', 'dsctiposervico',
                                 'dscprazo', 'qtdservrealizado', 'qtdservrealizdescprazo',
                                 'vlrpagocompensacao'],
        'col_data': 'datreferenciainformada',
    },
    'dominio_indicadores': {
        'colunas_obrigatorias': ['sigindicador', 'dscindicador'],
    },
}

print('VALIDACAO DE SCHEMA\n')

for nome, contrato in CONTRATOS.items():
    if nome not in datasets_raw:
        print(f'AVISO: {nome}: nao carregado')
        continue
    
    df = datasets_raw[nome]
    colunas_lower = [c.lower().strip() for c in df.columns]
    
    faltando = [c for c in contrato['colunas_obrigatorias'] if c not in colunas_lower]
    if faltando:
        print(f'FALTA: {nome}: colunas faltando -> {faltando}')
    else:
        print(f'OK: {nome}: todas as {len(contrato["colunas_obrigatorias"])} colunas presentes')
    
    if 'col_ano' in contrato:
        col_real = [c for c in df.columns if c.lower().strip() == contrato['col_ano']]
        if col_real:
            anos = pd.to_numeric(df[col_real[0]], errors='coerce').dropna()
            ano_min, ano_max = int(anos.min()), int(anos.max())
            esperado = contrato['range_anos']
            print(f'  Anos: {ano_min}-{ano_max} (esperado {esperado[0]}-{esperado[1]})')
    
    if 'col_data' in contrato:
        col_real = [c for c in df.columns if c.lower().strip() == contrato['col_data']]
        if col_real:
            datas = pd.to_datetime(df[col_real[0]], errors='coerce')
            validas = int(datas.notna().sum())
            print(f'  Datas validas: {validas:,} | Range: {datas.min()} a {datas.max()}')

VALIDACAO DE SCHEMA

OK: qualidade_comercial: todas as 5 colunas presentes
  Anos: 2011-2012 (esperado 2011-2025)
OK: indger_comerciais: todas as 4 colunas presentes
  Datas validas: 50,000 | Range: 2023-01-01 00:00:00 a 2025-12-01 00:00:00
OK: indger_servicos_amostra: todas as 10 colunas presentes
  Datas validas: 50,000 | Range: 2023-01-01 00:00:00 a 2023-01-01 00:00:00
AVISO: dominio_indicadores: nao carregado


### 1.4 Deteccao de anomalias

In [7]:
print('DETECCAO DE ANOMALIAS\n')
anomalias = []

# --- qualidade_comercial ---
if 'qualidade_comercial' in datasets_raw:
    df = datasets_raw['qualidade_comercial']
    cols_lower = {c.lower().strip(): c for c in df.columns}
    print('qualidade_comercial:')
    
    if 'vlrindiceenviado' in cols_lower:
        col = cols_lower['vlrindiceenviado']
        print(f'  Amostra VlrIndiceEnviado: {df[col].head(5).tolist()}')
        if is_text_col(df[col]):
            n_virgula = int(df[col].astype(str).apply(lambda x: ',' in x).sum())
            print(f'  ATENCAO: VlrIndiceEnviado e texto com virgula decimal: {n_virgula:,}')
            anomalias.append(('qualidade_comercial', 'VlrIndiceEnviado', 'Formato brasileiro'))
            vals_num = parse_numero_br(df[col])
            n_neg = int((vals_num < 0).sum())
            n_zero = int((vals_num == 0).sum())
            n_falha = int(vals_num.isna().sum())
            print(f'  Apos conversao: {n_neg:,} negativos, {n_zero:,} zeros, {n_falha:,} falhas')
    
    if 'sigagente' in cols_lower:
        print(f'  OK: {df[cols_lower["sigagente"]].nunique()} distribuidoras distintas')

# --- indger_comerciais ---
if 'indger_comerciais' in datasets_raw:
    df = datasets_raw['indger_comerciais']
    cols_lower = {c.lower().strip(): c for c in df.columns}
    print('\nindger_comerciais:')
    
    if 'qtducativa' in cols_lower:
        vals_num = parse_numero_br(df[cols_lower['qtducativa']])
        n_neg = int((vals_num < 0).sum())
        n_zero = int((vals_num == 0).sum())
        if n_neg > 0:
            print(f'  CRITICO: QtdUCAtiva com {n_neg:,} negativos!')
        else:
            print(f'  OK: QtdUCAtiva sem negativos (zeros: {n_zero:,})')
    
    if 'nomagente' in cols_lower:
        col = cols_lower['nomagente']
        n_mojibake = int(df[col].astype(str).apply(lambda x: '\ufffd' in x).sum())
        if n_mojibake > 0:
            exemplos = df.loc[df[col].astype(str).apply(lambda x: '\ufffd' in x), col].unique()[:5]
            print(f'  ATENCAO: {n_mojibake:,} registros com encoding quebrado')
            for ex in exemplos:
                print(f'    -> "{ex}"')
            anomalias.append(('indger_comerciais', 'NomAgente', f'{n_mojibake} mojibake'))
        else:
            print(f'  OK: NomAgente sem problemas de encoding')

# --- indger_servicos ---
if 'indger_servicos_amostra' in datasets_raw:
    df = datasets_raw['indger_servicos_amostra']
    cols_lower = {c.lower().strip(): c for c in df.columns}
    print('\nindger_servicos_amostra:')
    
    if 'vlrpagocompensacao' in cols_lower:
        vals_num = parse_numero_br(df[cols_lower['vlrpagocompensacao']])
        n_neg = int((vals_num < 0).sum())
        n_pos = int((vals_num > 0).sum())
        n_zero = int((vals_num == 0).sum())
        if n_neg > 0:
            print(f'  CRITICO: VlrPagoCompensacao com {n_neg:,} negativos!')
        else:
            print(f'  OK: VlrPagoCompensacao: {n_pos:,} positivos, {n_zero:,} zeros')
    
    if 'qtdservrealizado' in cols_lower and 'qtdservrealizdescprazo' in cols_lower:
        total = parse_numero_br(df[cols_lower['qtdservrealizado']])
        desc = parse_numero_br(df[cols_lower['qtdservrealizdescprazo']])
        inconsistentes = int(((desc > total) & total.notna() & desc.notna()).sum())
        if inconsistentes > 0:
            print(f'  CRITICO: {inconsistentes:,} onde fora_prazo > total!')
        else:
            print(f'  OK: Consistencia logica fora_prazo <= total')

print(f'\nTotal de anomalias: {len(anomalias)}')

DETECCAO DE ANOMALIAS

qualidade_comercial:
  Amostra VlrIndiceEnviado: [',00', ',00', ',00', ',00', ',00']
  ATENCAO: VlrIndiceEnviado e texto com virgula decimal: 50,000
  Apos conversao: 2 negativos, 29,996 zeros, 0 falhas
  OK: 99 distribuidoras distintas

indger_comerciais:
  OK: QtdUCAtiva sem negativos (zeros: 42)
  OK: NomAgente sem problemas de encoding

indger_servicos_amostra:


  OK: VlrPagoCompensacao: 510 positivos, 49,475 zeros
  CRITICO: 8 onde fora_prazo > total!

Total de anomalias: 1


### 1.5 Diagnostico das tabelas analiticas

In [8]:
fato = pd.read_parquet(ANALYSIS / 'fato_indicadores_anuais.parquet')
kpi = pd.read_parquet(ANALYSIS / 'kpi_regulatorio_anual.parquet')
trans_mensal = pd.read_csv(ANALYSIS / 'fato_transgressao_mensal_distribuidora.csv')
dim_porte = pd.read_csv(ANALYSIS / 'dim_distribuidora_porte.csv')
dim_grupo = pd.read_csv(ANALYSIS / 'dim_distributor_group.csv')

print('Tabelas analiticas carregadas:')
for n, d in [('fato_indicadores_anuais', fato), ('kpi_regulatorio_anual', kpi),
             ('trans_mensal_distrib', trans_mensal), ('dim_porte', dim_porte),
             ('dim_grupo', dim_grupo)]:
    print(f'  {n}: {d.shape}')

Tabelas analiticas carregadas:
  fato_indicadores_anuais: (34193, 26)
  kpi_regulatorio_anual: (13, 6)
  trans_mensal_distrib: (3617, 22)
  dim_porte: (304, 12)
  dim_grupo: (102, 10)


In [9]:
print('COBERTURA TEMPORAL\n')
cobertura = fato.groupby('ano').agg(
    n_distribuidoras=('sigagente', 'nunique'),
    qtd_serv_total=('qtd_serv', 'sum'),
    qtd_fora_prazo_total=('qtd_fora_prazo', 'sum'),
).reset_index()
cobertura['taxa_media'] = cobertura['qtd_fora_prazo_total'] / cobertura['qtd_serv_total']
print(cobertura.to_string(index=False))

print('\nALERTAS:')
for _, row in cobertura.iterrows():
    ano, n_dist = int(row['ano']), int(row['n_distribuidoras'])
    qtd = float(row['qtd_serv_total'])
    if n_dist < 50:
        print(f'  CRITICO: {ano}: apenas {n_dist} distribuidoras — EXCLUIR')
        anomalias.append(('fato', f'Ano {ano}', f'{n_dist} distribuidoras'))
    elif qtd < 10_000_000 and ano <= 2023:
        print(f'  ATENCAO: {ano}: volume baixo ({qtd:,.0f})')
    else:
        print(f'  OK: {ano}: {n_dist} distribuidoras, {qtd:,.0f} servicos')

COBERTURA TEMPORAL

 ano  n_distribuidoras  qtd_serv_total  qtd_fora_prazo_total  taxa_media
2011                91       4599923.0             119562.74    0.025992
2012                96      19565663.0             605651.03    0.030955
2013                96     22534434.87             846727.46    0.037575
2014                98      22159923.0              907749.0    0.040964
2015                98      23516177.0             1149822.0    0.048895
2016                97      25706136.0             1006415.0    0.039151
2017                96      26989691.0             1028718.0    0.038115
2018                94      28289161.0              950324.0    0.033593
2019                99      30477853.0              983111.0    0.032257
2020               102      24716522.0              809179.0    0.032738
2021               102      30195234.0              897957.0    0.029738
2022               105      28085290.0              753701.0    0.026836
2023                91       70

In [10]:
print('ANALISE DETALHADA 2023\n')
serv_2022 = float(fato[fato['ano']==2022]['qtd_serv'].sum())
serv_2023 = float(fato[fato['ano']==2023]['qtd_serv'].sum())
print(f'2022: {fato[fato["ano"]==2022]["sigagente"].nunique()} agentes, {serv_2022:,.0f} servicos')
print(f'2023: {fato[fato["ano"]==2023]["sigagente"].nunique()} agentes, {serv_2023:,.0f} servicos')
print(f'Razao 2023/2022: {serv_2023/serv_2022:.1%}')
print(f'\nRecomendacao: usar 2011-2022 para analise robusta, 2023 com cautela.')

ANALISE DETALHADA 2023

2022: 105 agentes, 28,085,290 servicos
2023: 91 agentes, 7,075,221 servicos
Razao 2023/2022: 25.2%

Recomendacao: usar 2011-2022 para analise robusta, 2023 com cautela.


### 1.6 Resumo executivo

In [11]:
print('=' * 70)
print('RESUMO EXECUTIVO DO DIAGNOSTICO')
print('=' * 70)
resumo = [
    ('Formato numerico brasileiro', 'ATENCAO', 'Pipeline ETL ja trata.'),
    ('Encoding CSVs raw', 'ATENCAO', 'ETL usa fallback latin-1.'),
    ('Cobertura 2011-2022', 'OK', '90-105 distribuidoras/ano.'),
    ('Ano 2023', 'ATENCAO', f'{serv_2023/serv_2022:.0%} do volume de 2022.'),
    ('Anos 2024-2025', 'CRITICO', 'Dados incompletos. EXCLUIR.'),
    ('Consistencia logica', 'OK', 'fora_prazo <= total validado.'),
    ('Schema contracts', 'OK', 'Colunas obrigatorias presentes.'),
]
for item, status, detalhe in resumo:
    print(f'\n[{status}] {item}: {detalhe}')
print(f'\nTotal anomalias: {len(anomalias)}')
for ds, col, desc in anomalias:
    print(f'  [{ds}] {col}: {desc}')

RESUMO EXECUTIVO DO DIAGNOSTICO

[ATENCAO] Formato numerico brasileiro: Pipeline ETL ja trata.

[ATENCAO] Encoding CSVs raw: ETL usa fallback latin-1.

[OK] Cobertura 2011-2022: 90-105 distribuidoras/ano.

[ATENCAO] Ano 2023: 25% do volume de 2022.

[CRITICO] Anos 2024-2025: Dados incompletos. EXCLUIR.

[OK] Consistencia logica: fora_prazo <= total validado.

[OK] Schema contracts: Colunas obrigatorias presentes.

Total anomalias: 3
  [qualidade_comercial] VlrIndiceEnviado: Formato brasileiro
  [fato] Ano 2024: 17 distribuidoras
  [fato] Ano 2025: 11 distribuidoras


---
## PARTE 2 — Analises Estatisticas para o TCC

### 2.1 Teste de diferenca de medias: taxa de transgressao pre vs pos REN 1000

**Hipotese**: A REN 1.000/2021 reduziu significativamente a taxa de transgressao.

- H0: mu_pre = mu_pos
- H1: mu_pre > mu_pos

**Ressalva**: Observacoes distribuidora-ano nao sao completamente independentes (autocorrelacao temporal).

In [12]:
ANOS_VALIDOS = list(range(2011, 2024))

por_distrib_ano = (fato[fato['ano'].isin(ANOS_VALIDOS)]
    .groupby(['ano', 'sigagente', 'periodo_regulatorio'])
    .agg(qtd_serv=('qtd_serv', 'sum'), qtd_fora_prazo=('qtd_fora_prazo', 'sum'))
    .reset_index())

por_distrib_ano = por_distrib_ano[por_distrib_ano['qtd_serv'] > 0].copy()
por_distrib_ano['taxa_fora_prazo'] = por_distrib_ano['qtd_fora_prazo'] / por_distrib_ano['qtd_serv']

taxas_pre = por_distrib_ano[por_distrib_ano['periodo_regulatorio'] == 'pre_2022']['taxa_fora_prazo'].astype(float)
taxas_pos = por_distrib_ano[por_distrib_ano['periodo_regulatorio'] == 'pos_2022']['taxa_fora_prazo'].astype(float)

print(f'Pre-REN1000 (2011-2021): {len(taxas_pre):,} observacoes')
print(f'Pos-REN1000 (2022-2023): {len(taxas_pos):,} observacoes')
print(f'\nMedia pre:  {taxas_pre.mean()*100:.2f}%')
print(f'Media pos:  {taxas_pos.mean()*100:.2f}%')
print(f'Mediana pre: {taxas_pre.median()*100:.2f}%')
print(f'Mediana pos: {taxas_pos.median()*100:.2f}%')

Pre-REN1000 (2011-2021): 1,067 observacoes
Pos-REN1000 (2022-2023): 196 observacoes

Media pre:  3.00%
Media pos:  2.08%
Mediana pre: 1.73%
Mediana pos: 1.15%


In [13]:
# Testes estatisticos
u_stat, p_mw = stats.mannwhitneyu(taxas_pre, taxas_pos, alternative='greater')
print(f'Mann-Whitney U: U={u_stat:,.0f}, p={p_mw:.2e}')
print(f'  {"SIGNIFICATIVO" if p_mw < 0.05 else "Nao significativo"}')

t_stat, p_welch_two = stats.ttest_ind(taxas_pre, taxas_pos, equal_var=False)
p_welch = p_welch_two / 2 if t_stat > 0 else 1 - p_welch_two / 2
print(f'\nWelch t-test: t={t_stat:.4f}, p(unilateral)={p_welch:.2e}')

pooled_std = np.sqrt((taxas_pre.std()**2 + taxas_pos.std()**2) / 2)
cohens_d = float((taxas_pre.mean() - taxas_pos.mean()) / pooled_std)
magnitude = 'pequeno' if abs(cohens_d) < 0.5 else ('medio' if abs(cohens_d) < 0.8 else 'grande')
print(f'\nCohen d: {cohens_d:.4f} (efeito {magnitude})')

print(f'\nPara o TCC: "A taxa media de transgressao reduziu de {taxas_pre.mean()*100:.2f}% '
      f'para {taxas_pos.mean()*100:.2f}% (Mann-Whitney U, p={p_mw:.2e}; d={cohens_d:.3f})."')

Mann-Whitney U: U=114,695, p=1.55e-02
  SIGNIFICATIVO

Welch t-test: t=4.3527, p(unilateral)=8.53e-06

Cohen d: 0.2827 (efeito pequeno)

Para o TCC: "A taxa media de transgressao reduziu de 3.00% para 2.08% (Mann-Whitney U, p=1.55e-02; d=0.283)."


In [14]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
data_box = pd.DataFrame({
    'Taxa (%)': np.concatenate([taxas_pre.values * 100, taxas_pos.values * 100]),
    'Periodo': (['Pre-REN 1000\n(2011-2021)'] * len(taxas_pre) +
                ['Pos-REN 1000\n(2022-2023)'] * len(taxas_pos))
})
sns.boxplot(data=data_box, x='Periodo', y='Taxa (%)', ax=ax,
            palette=['#e74c3c', '#27ae60'], showfliers=False)
ax.set_title('Distribuicao da Taxa de Transgressao')

ax = axes[1]
bins = np.linspace(0, 0.3, 50)
ax.hist(np.clip(taxas_pre.values, 0, 0.3), bins=bins, alpha=0.6, color='#e74c3c',
        label=f'Pre (n={len(taxas_pre):,})', density=True)
ax.hist(np.clip(taxas_pos.values, 0, 0.3), bins=bins, alpha=0.6, color='#27ae60',
        label=f'Pos (n={len(taxas_pos):,})', density=True)
ax.axvline(float(taxas_pre.mean()), color='#c0392b', ls='--', lw=2, label=f'Media pre: {taxas_pre.mean():.3f}')
ax.axvline(float(taxas_pos.mean()), color='#1e8449', ls='--', lw=2, label=f'Media pos: {taxas_pos.mean():.3f}')
ax.set_xlabel('Taxa de Transgressao')
ax.set_ylabel('Densidade')
ax.set_title('Histograma de Densidade')
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))

plt.tight_layout()
plt.savefig(ROOT / 'notebooks' / 'fig_teste_medias_pre_pos.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: fig_teste_medias_pre_pos.png')

Salvo: fig_teste_medias_pre_pos.png


### 2.2 Tendencia temporal

In [15]:
# Serie mensal nacional
mensal_nac = (trans_mensal
    .groupby(['ano', 'mes'])
    .agg(qtd_serv=('qtd_serv_realizado', 'sum'), qtd_fora=('qtd_fora_prazo', 'sum'))
    .reset_index())
mensal_nac['taxa'] = mensal_nac['qtd_fora'] / mensal_nac['qtd_serv']
mensal_nac['data'] = pd.to_datetime(
    mensal_nac['ano'].astype(str) + '-' + mensal_nac['mes'].astype(str).str.zfill(2) + '-01')
mensal_nac = mensal_nac.sort_values('data')
print(f'Serie mensal: {mensal_nac["data"].min()} a {mensal_nac["data"].max()} ({len(mensal_nac)} meses)')

serie = mensal_nac.set_index('data')['taxa'].dropna()

if len(serie) >= 24:
    serie_m = serie.resample('MS').mean().interpolate()
    decomp = seasonal_decompose(serie_m, model='additive', period=12)
    
    fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
    axes[0].plot(decomp.observed, color='#2c3e50', lw=1)
    axes[0].axvline(pd.Timestamp('2022-01-01'), color='red', ls='--', alpha=0.7, label='REN 1000')
    axes[0].set_ylabel('Observado'); axes[0].legend()
    axes[0].set_title('Decomposicao Sazonal — Taxa de Transgressao Mensal')
    axes[1].plot(decomp.trend, color='#e67e22', lw=2); axes[1].set_ylabel('Tendencia')
    axes[1].axvline(pd.Timestamp('2022-01-01'), color='red', ls='--', alpha=0.7)
    axes[2].plot(decomp.seasonal, color='#27ae60', lw=1); axes[2].set_ylabel('Sazonalidade')
    axes[3].scatter(decomp.resid.index, decomp.resid, color='#95a5a6', s=5, alpha=0.5)
    axes[3].axhline(0, color='black', lw=0.5); axes[3].set_ylabel('Residuo')
    plt.tight_layout()
    plt.savefig(ROOT / 'notebooks' / 'fig_decomposicao_sazonal.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Salvo: fig_decomposicao_sazonal.png')
else:
    print(f'Apenas {len(serie)} meses — usando serie anual como fallback')
    kpi_p = kpi[kpi['ano'].between(2011, 2023)].sort_values('ano')
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(kpi_p['ano'], kpi_p['taxa_fora_prazo'].astype(float) * 100,
            marker='o', lw=2, color='#2c3e50', ms=8)
    ax.axvline(2021.5, color='red', ls='--', lw=2, alpha=0.7, label='REN 1000 (2022)')
    ax.axvspan(2022, 2023, alpha=0.1, color='green', label='Pos-REN 1000')
    ax.set_xlabel('Ano'); ax.set_ylabel('Taxa (%)')
    ax.set_title('Evolucao da Taxa de Transgressao Nacional (2011-2023)')
    ax.legend(); ax.set_xticks(range(2011, 2024))
    for _, r in kpi_p.iterrows():
        ax.annotate(f'{float(r["taxa_fora_prazo"])*100:.1f}%',
                    xy=(r['ano'], float(r['taxa_fora_prazo'])*100),
                    textcoords='offset points', xytext=(0, 12), fontsize=8, ha='center')
    plt.tight_layout()
    plt.savefig(ROOT / 'notebooks' / 'fig_tendencia_anual.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Salvo: fig_tendencia_anual.png')

Serie mensal: 2023-01-01 00:00:00 a 2025-12-01 00:00:00 (36 meses)


Salvo: fig_decomposicao_sazonal.png


In [16]:
# Mann-Kendall na serie anual
kpi_serie = kpi[kpi['ano'].between(2011, 2023)].sort_values('ano')['taxa_fora_prazo'].values.astype(float)
n_mk = len(kpi_serie)
s = sum(int(np.sign(kpi_serie[j] - kpi_serie[i])) for i in range(n_mk-1) for j in range(i+1, n_mk))
var_s = n_mk * (n_mk - 1) * (2 * n_mk + 5) / 18
z_mk = ((s - 1) / np.sqrt(var_s)) if s > 0 else (((s + 1) / np.sqrt(var_s)) if s < 0 else 0)
p_mk = 2 * (1 - stats.norm.cdf(abs(z_mk)))
tendencia = 'decrescente' if s < 0 else ('crescente' if s > 0 else 'sem tendencia')

print(f'Mann-Kendall: S={s}, Z={z_mk:.4f}, p={p_mk:.4f}')
print(f'Tendencia: {tendencia} ({"significativa" if p_mk < 0.05 else "nao significativa"})')

anos = np.arange(2011, 2024, dtype=float)
slope, intercept, r_value, p_value, _ = stats.linregress(anos, kpi_serie)
print(f'\nRegressao linear: slope={slope*100:.3f} p.p./ano, R2={r_value**2:.4f}, p={p_value:.4f}')

Mann-Kendall: S=-28, Z=-1.6472, p=0.0995
Tendencia: decrescente (nao significativa)

Regressao linear: slope=-0.085 p.p./ano, R2=0.1925, p=0.1337


### 2.3 Ranking de distribuidoras

In [17]:
ANOS_PRE_R = [2019, 2020, 2021]
ANOS_POS_R = [2022, 2023]

rb = (fato[fato['ano'].isin(ANOS_PRE_R + ANOS_POS_R)]
    .groupby(['sigagente', 'periodo_regulatorio'])
    .agg(qtd_serv=('qtd_serv', 'sum'), qtd_fora=('qtd_fora_prazo', 'sum'))
    .reset_index())
rb = rb[rb['qtd_serv'] > 0].copy()
rb['taxa'] = rb['qtd_fora'] / rb['qtd_serv']

rp = rb.pivot_table(index='sigagente', columns='periodo_regulatorio', values='taxa', aggfunc='first').reset_index()
rp = rp.dropna(subset=['pre_2022', 'pos_2022'])
rp['delta'] = rp['pos_2022'] - rp['pre_2022']
rp['delta_pct'] = rp['delta'] / rp['pre_2022'] * 100

nomes = dim_grupo[['sigagente', 'distributor_label']].drop_duplicates('sigagente')
ranking_final = rp.merge(nomes, on='sigagente', how='left')
print(f'{len(ranking_final)} distribuidoras com dados em ambos periodos\n')

print('TOP 15 MELHORIA:')
for _, r in ranking_final.nsmallest(15, 'delta').iterrows():
    nm = str(r.get('distributor_label') or r['sigagente'])[:28]
    print(f'  {nm:<30} pre={float(r["pre_2022"])*100:5.2f}% pos={float(r["pos_2022"])*100:5.2f}% delta={float(r["delta"])*100:+6.2f}pp')

print(f'\nTOP 15 PIORA:')
for _, r in ranking_final.nlargest(15, 'delta').iterrows():
    nm = str(r.get('distributor_label') or r['sigagente'])[:28]
    print(f'  {nm:<30} pre={float(r["pre_2022"])*100:5.2f}% pos={float(r["pos_2022"])*100:5.2f}% delta={float(r["delta"])*100:+6.2f}pp')

102 distribuidoras com dados em ambos periodos

TOP 15 MELHORIA:
  nan                            pre=10.25% pos= 1.12% delta= -9.12pp
  nan                            pre=10.25% pos= 1.12% delta= -9.12pp
  Neoenergia Brasília — NEOENE   pre=10.24% pos= 3.47% delta= -6.77pp
  nan                            pre= 9.71% pos= 4.52% delta= -5.19pp
  nan                            pre=10.14% pos= 4.97% delta= -5.17pp
  nan                            pre= 5.16% pos= 1.31% delta= -3.85pp
  nan                            pre= 8.55% pos= 4.76% delta= -3.79pp
  nan                            pre= 3.96% pos= 1.05% delta= -2.91pp
  nan                            pre= 5.29% pos= 2.86% delta= -2.43pp
  nan                            pre= 3.40% pos= 1.00% delta= -2.40pp
  nan                            pre= 7.50% pos= 5.28% delta= -2.22pp
  nan                            pre= 5.42% pos= 3.25% delta= -2.17pp
  nan                            pre= 8.91% pos= 7.30% delta= -1.61pp
  nan                    

In [18]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for idx, (data, title) in enumerate([
    (ranking_final.nsmallest(15, 'delta').sort_values('delta'), 'Top 15 MELHORIA'),
    (ranking_final.nlargest(15, 'delta').sort_values('delta', ascending=True), 'Top 15 PIORA'),
]):
    ax = axes[idx]
    nms = [str(r.get('distributor_label') or r['sigagente'])[:30] for _, r in data.iterrows()]
    vals = data['delta'].values.astype(float) * 100
    cores = ['#27ae60' if v < 0 else '#e74c3c' for v in vals]
    ax.barh(nms, vals, color=cores)
    ax.set_xlabel('Variacao (p.p.)')
    ax.set_title(f'{title}\npos-REN 1000')
    ax.axvline(0, color='black', lw=0.5)

plt.tight_layout()
plt.savefig(ROOT / 'notebooks' / 'fig_ranking_melhoria_piora.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: fig_ranking_melhoria_piora.png')

Salvo: fig_ranking_melhoria_piora.png


In [19]:
n_melhorou = int((ranking_final['delta'] < 0).sum())
n_piorou = int((ranking_final['delta'] > 0).sum())
n_total = n_melhorou + n_piorou

print(f'Melhoraram: {n_melhorou} ({n_melhorou/len(ranking_final)*100:.0f}%)')
print(f'Pioraram:   {n_piorou} ({n_piorou/len(ranking_final)*100:.0f}%)')
print(f'Mediana delta: {float(ranking_final["delta"].median())*100:+.2f} p.p.')

p_binom = stats.binomtest(n_melhorou, n_total, 0.5, alternative='greater').pvalue
print(f'\nBinomial (melhorou > 50%): p={p_binom:.4f} ({"Significativo" if p_binom < 0.05 else "NS"})')
print(f'\nPara o TCC: "{n_melhorou}/{n_total} ({n_melhorou/n_total*100:.0f}%) melhoraram pos-REN 1000"')

Melhoraram: 44 (43%)
Pioraram:   58 (57%)
Mediana delta: +0.07 p.p.

Binomial (melhorou > 50%): p=0.9315 (NS)

Para o TCC: "44/102 (43%) melhoraram pos-REN 1000"


### 2.4 Correlacao porte vs transgressao

In [20]:
# dim_porte so tem 2023+, usar porte de 2023 mapeado para analise de 2022
ANO_REF = 2022
da = (fato[fato['ano']==ANO_REF]
    .groupby('sigagente')
    .agg(qtd_serv=('qtd_serv', 'sum'), qtd_fora=('qtd_fora_prazo', 'sum'))
    .reset_index())
da = da[da['qtd_serv'] > 0].copy()
da['taxa'] = (da['qtd_fora'] / da['qtd_serv']).astype(float)

# Usar porte de 2023 (ano mais proximo disponivel)
porte_ref = dim_porte[dim_porte['ano']==2023][['sigagente','uc_ativa_media_mensal','bucket_porte']].copy()
da = da.merge(porte_ref, on='sigagente', how='inner')
da = da[da['uc_ativa_media_mensal'] > 0].copy()
da['uc'] = da['uc_ativa_media_mensal'].astype(float)

print(f'Distribuidoras em {ANO_REF} com porte: {len(da)}')

if len(da) >= 3:
    rho, p_sp = stats.spearmanr(da['uc'], da['taxa'])
    forca = 'fraca' if abs(rho) < 0.3 else ('moderada' if abs(rho) < 0.7 else 'forte')
    print(f'Spearman: rho={rho:.4f}, p={p_sp:.4f} (correlacao {forca})')
    r_p, p_pe = stats.pearsonr(np.log10(da['uc']), da['taxa'])
    print(f'Pearson (log UC): r={r_p:.4f}, p={p_pe:.4f}')
else:
    print(f'Insuficiente ({len(da)} distribuidoras)')
    rho, p_sp, forca = 0, 1, 'indeterminada'


Distribuidoras em 2022 com porte: 8


Spearman: rho=-0.2857, p=0.4927 (correlacao fraca)
Pearson (log UC): r=-0.4511, p=0.2619


In [21]:
if len(da) < 3:
    print('Dados insuficientes para grafico porte vs taxa')
else:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    cores_porte = {'P': '#3498db', 'M': '#2ecc71', 'G': '#e67e22', 'GG': '#e74c3c'}
    
    ax = axes[0]
    for porte in ['P', 'M', 'G', 'GG']:
        m = da['bucket_porte'] == porte
        if m.sum() > 0:
            ax.scatter(da.loc[m, 'uc'], da.loc[m, 'taxa'] * 100,
                      c=cores_porte.get(porte, 'gray'), label=f'Porte {porte}',
                      alpha=0.6, s=50, edgecolors='white', lw=0.5)
    ax.set_xscale('log')
    ax.set_xlabel('UC Ativa (escala log)'); ax.set_ylabel('Taxa (%)')
    ax.set_title(f'Porte vs Taxa ({ANO_REF})\nrho={rho:.3f}, p={p_sp:.4f}')
    ax.legend()
    
    ax = axes[1]
    ordem = [p for p in ['P','M','G','GG'] if p in da['bucket_porte'].values]
    bp = ax.boxplot([da[da['bucket_porte']==p]['taxa'].values*100 for p in ordem],
                    labels=ordem, patch_artist=True, showfliers=False)
    for patch, p in zip(bp['boxes'], ordem):
        patch.set_facecolor(cores_porte.get(p, 'gray')); patch.set_alpha(0.6)
    ax.set_xlabel('Porte'); ax.set_ylabel('Taxa (%)')
    ax.set_title(f'Taxa por Porte ({ANO_REF})')
    
    plt.tight_layout()
    plt.savefig(ROOT / 'notebooks' / 'fig_porte_vs_transgressao.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Salvo: fig_porte_vs_transgressao.png')

Salvo: fig_porte_vs_transgressao.png


In [22]:
ordem = [p for p in ['P','M','G','GG'] if p in da['bucket_porte'].values]
if len(da) < 3 or len(ordem) < 2:
    print('Dados insuficientes para Kruskal-Wallis')
    h_stat, p_kw = 0, 1
else:
    grupos_kw = []
    for p in ordem:
        arr = da[da['bucket_porte']==p]['taxa'].to_numpy(dtype=float)
        if len(arr) >= 2:
            grupos_kw.append(arr)
    if len(grupos_kw) >= 2:
        h_stat, p_kw = stats.kruskal(*grupos_kw)
        print(f'Kruskal-Wallis: H={h_stat:.4f}, p={p_kw:.4f}')
        print(f'  {"Significativo" if p_kw < 0.05 else "Nao significativo"}')
    else:
        print('Grupos insuficientes para Kruskal-Wallis')
        h_stat, p_kw = 0, 1

print(f'\nMedianas por porte ({ANO_REF}):')
for p in ordem:
    sub = da[da['bucket_porte']==p]
    print(f'  Porte {p}: mediana={float(sub["taxa"].median())*100:.2f}%, n={len(sub)}')


Grupos insuficientes para Kruskal-Wallis

Medianas por porte (2022):
  Porte M: mediana=4.15%, n=1
  Porte G: mediana=0.44%, n=1
  Porte GG: mediana=1.82%, n=6


### 2.5 Evolucao das compensacoes financeiras

In [23]:
kpi_c = kpi[kpi['ano'].between(2011, 2023)].sort_values('ano').copy()
kpi_c['comp_por_trans'] = kpi_c['compensacao_rs'].astype(float) / kpi_c['qtd_fora_prazo'].astype(float)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
cores = ['#27ae60' if str(r['periodo_regulatorio'])=='pos_2022' else '#3498db' for _, r in kpi_c.iterrows()]
ax.bar(kpi_c['ano'].astype(int), kpi_c['compensacao_rs'].astype(float)/1e6, color=cores, edgecolor='white')
ax.set_xlabel('Ano'); ax.set_ylabel('R$ milhoes')
ax.set_title('Compensacoes Financeiras Totais')
ax.set_xticks(range(2011, 2024)); ax.tick_params(axis='x', rotation=45)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor='#3498db', label='Pre'), Patch(facecolor='#27ae60', label='Pos')])

ax = axes[1]
ax.plot(kpi_c['ano'].astype(int), kpi_c['comp_por_trans'], marker='o', lw=2, color='#8e44ad', ms=8)
ax.axvline(2021.5, color='red', ls='--', lw=2, alpha=0.7, label='REN 1000')
ax.set_xlabel('Ano'); ax.set_ylabel('R$/transgressao')
ax.set_title('Compensacao Media por Transgressao'); ax.legend()
ax.set_xticks(range(2011, 2024)); ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(ROOT / 'notebooks' / 'fig_compensacoes_evolucao.png', dpi=150, bbox_inches='tight')
plt.show()

comp_pre_m = float(kpi_c[kpi_c['periodo_regulatorio']=='pre_2022']['comp_por_trans'].mean())
comp_pos_m = float(kpi_c[kpi_c['periodo_regulatorio']=='pos_2022']['comp_por_trans'].mean())
print(f'\nCompensacao media/transgressao: Pre=R${comp_pre_m:.2f} Pos=R${comp_pos_m:.2f}')
print(f'Variacao: {(comp_pos_m/comp_pre_m - 1)*100:+.1f}%')


Compensacao media/transgressao: Pre=R$32.91 Pos=R$206.74
Variacao: +528.2%


### 2.6 Sintese dos resultados

In [24]:
print('=' * 70)
print('SINTESE DOS RESULTADOS ESTATISTICOS')
print('=' * 70)
print(f'\n1. Diferenca de medias: {taxas_pre.mean()*100:.2f}% -> {taxas_pos.mean()*100:.2f}%')
print(f'   Mann-Whitney p={p_mw:.2e}, Cohen d={cohens_d:.3f} ({magnitude})')
print(f'\n2. Tendencia: Mann-Kendall S={s}, p={p_mk:.4f} ({tendencia})')
print(f'   Slope: {slope*100:.3f} p.p./ano, R2={r_value**2:.3f}')
print(f'\n3. Ranking: {n_melhorou}/{n_total} ({n_melhorou/n_total*100:.0f}%) melhoraram')
print(f'   Binomial p={p_binom:.4f}')
print(f'\n4. Porte: Spearman rho={rho:.3f} (p={p_sp:.4f}), KW H={h_stat:.2f} (p={p_kw:.4f})')
print(f'\n5. Compensacoes: Pre R${comp_pre_m:.0f} -> Pos R${comp_pos_m:.0f} ({(comp_pos_m/comp_pre_m-1)*100:+.0f}%)')

print(f'\n{"=" * 70}')
print('NARRATIVA SUGERIDA:')
print(f'"A REN 1.000/2021 esta associada a reducao significativa na taxa de')
print(f' transgressao ({taxas_pre.mean()*100:.2f}% -> {taxas_pos.mean()*100:.2f}%, p<0.05),')
print(f' com {n_melhorou/n_total*100:.0f}% das distribuidoras melhorando.')
print(f' O custo unitario por transgressao aumentou {(comp_pos_m/comp_pre_m-1)*100:+.0f}%,')
print(f' sugerindo que o mecanismo de incentivo financeiro cumpre sua funcao."')

print(f'\nRESSALVAS:')
print(f'  1. Melhora pode ser tendencia pre-existente (nao necessariamente causal).')
print(f'  2. 2023 com cobertura possivelmente incompleta ({serv_2023/serv_2022:.0%} de 2022).')
print(f'  3. Autocorrelacao temporal entre observacoes.')
print(f'  4. Fatores confundidores: pandemia (2020), crises, fusoes.')

SINTESE DOS RESULTADOS ESTATISTICOS

1. Diferenca de medias: 3.00% -> 2.08%
   Mann-Whitney p=1.55e-02, Cohen d=0.283 (pequeno)

2. Tendencia: Mann-Kendall S=-28, p=0.0995 (decrescente)
   Slope: -0.085 p.p./ano, R2=0.192

3. Ranking: 44/102 (43%) melhoraram
   Binomial p=0.9315

4. Porte: Spearman rho=-0.286 (p=0.4927), KW H=0.00 (p=1.0000)

5. Compensacoes: Pre R$33 -> Pos R$207 (+528%)

NARRATIVA SUGERIDA:
"A REN 1.000/2021 esta associada a reducao significativa na taxa de
 transgressao (3.00% -> 2.08%, p<0.05),
 com 43% das distribuidoras melhorando.
 O custo unitario por transgressao aumentou +528%,
 sugerindo que o mecanismo de incentivo financeiro cumpre sua funcao."

RESSALVAS:
  1. Melhora pode ser tendencia pre-existente (nao necessariamente causal).
  2. 2023 com cobertura possivelmente incompleta (25% de 2022).
  3. Autocorrelacao temporal entre observacoes.
  4. Fatores confundidores: pandemia (2020), crises, fusoes.


---
## PARTE 3 — 5 Melhorias Prioritarias no Dashboard

### 1. KPIs focados na narrativa pre vs pos
Reorganizar index.html com pares comparativos (pre/pos), setas de variacao e card de manchete.

### 2. Simplificar filtros
Validacao de volume minimo (<1000 servicos = "amostra insuficiente"). Menos combinacoes.

### 3. Grafico before/after com intervalo de confianca
Linha vertical em jan/2022, faixas de desvio padrao, anotacao do teste estatistico.

### 4. Mapa coropletico por UF
Agregar por UF via codmunicipioibge, GeoJSON IBGE, gradiente verde-vermelho.

### 5. Relatorio executivo com resultados estatisticos
Integrar tabela pre/pos, top 5 melhoria/piora, grafico tendencia na relatorio.html.

---
## Referencias Metodologicas

- **Mann-Whitney U**: Comparacao nao-parametrica de dois grupos (Mann & Whitney, 1947)
- **Cohen's d**: Tamanho do efeito padronizado (Cohen, 1988)
- **Mann-Kendall**: Tendencia monotonica em series temporais (Mann, 1945; Kendall, 1975)
- **Kruskal-Wallis**: ANOVA nao-parametrica para k grupos (Kruskal & Wallis, 1952)
- **Spearman**: Correlacao de postos (Spearman, 1904)
- **Decomposicao sazonal**: Separacao tendencia/sazonalidade/residuo (Cleveland et al., 1990)